In [1]:
# Install dependencies
%pip install anthropic python-dotenv


[notice] A new release of pip is available: 26.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
#Load environment variables
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
# Create an API client
from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5"

In [25]:
def add_user_message(messages, text):
    user_message = {
        "role": "user",
        "content": text
    }
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {
        "role": "assistant",
        "content": text
    }
    messages.append(assistant_message)

# Make a request
def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }

    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    if system:
        params["system"] = system
        
    message = client.messages.create(**params)
    return message.content[0].text


In [5]:
import json

def generate_dataset():
    prompt = """
    Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompt 
    that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON
    each representing task that requires Python, JSON, or a Regex to complete.
    
    Example output:
    ```json
    [
        {
            "task": "Description of the task",
        },
        ...additional
    ] 
    ``` 

    * Focus on tasks that can be solved writing a single Python function, a single JSON object,
    * Focus on tasks that do not require writing much code

    Please generate 3 objects
    """

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [6]:
dataset = generate_dataset()

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [7]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
    Please solve the following task:
    
    {test_case["task"]}
    """

    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [26]:
def grade_by_model(test_case, output):
    # Create evaluation prompt
    eval_prompt = f"""
    You are an expert code reviewer. Evaluate this AI-generated solution.
    
    Task: {test_case["task"]}
    Solution: {output}
    
    Provide your evaluation as a structured JSON object with:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement  
    - "reasoning": A concise explanation of your assessment
    - "score": A number between 1-10
    """
    
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [27]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    # TODO - Grading
    model_grade = grade_by_model(test_case, output)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
    }


In [33]:
from statistics import mean


def run_eval(dataset):
    """Loads the dataset and calls run_test_case for each test case, then returns the results"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average Score: {average_score:.2f}")
        
    return results

In [34]:
with open("dataset.json") as f:
    dataset = json.load(f)
    
results = run_eval(dataset)

Average Score: 7.00


In [35]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS S3 Bucket Name Validator\n\n```python\nimport re\n\ndef is_valid_s3_bucket_name(bucket_name: str) -> bool:\n    \"\"\"\n    Validates if a given string is a valid AWS S3 bucket name according to AWS naming rules.\n    \n    AWS S3 Bucket Naming Rules:\n    - Must be between 3 and 63 characters long\n    - Can only contain lowercase letters, numbers, and hyphens\n    - Must start with a lowercase letter or number\n    - Must end with a lowercase letter or number\n    - Cannot contain consecutive hyphens\n    - Cannot be formatted as an IP address (e.g., 192.168.1.1)\n    \n    Args:\n        bucket_name (str): The bucket name to validate\n        \n    Returns:\n        bool: True if the bucket name is valid, False otherwise\n    \"\"\"\n    \n    # Check if bucket_name is a string\n    if not isinstance(bucket_name, str):\n        return False\n    \n    # Check length (3-63 characters)\n    if len(bucket_name) < 3 or len(bucket_name) > 63:\n        return Fa